# EmotionSense AI — Deep Learning Model Training
### High-Accuracy Facial Emotion Recognition (FER2013)

This notebook trains a **Deep Mini-Xception Network** with residual skip connections on the FER2013 dataset.

#### Key Techniques to Boost Accuracy:
1. **Balanced Class Weighting:** Solves the 16x class imbalance between `happy` (7,215) and `disgust` (436).
2. **Real-time Data Augmentation:** Prevents overfitting on low-resolution 48x48 images (flips, rotations, zoom, translation).
3. **Depthwise Separable Convolutions & Skip Connections:** Drastically reduces parameters from millions down to ~200K while retaining deep spatial features.
4. **Global Average Pooling:** Eliminates dense fully-connected layers that cause severe overfitting.
5. **Label Smoothing (0.08):** Prevents overconfidence on noisy/ambiguous human facial annotations.
6. **Dynamic Learning Rate Decay:** Reduces learning rate when validation loss plateaus.

In [ ]:
# Step 1: Check GPU Acceleration
import tensorflow as tf
print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f' SUCCESS: GPU Detected -> {gpus[0]}')
else:
    print(' WARNING: No GPU detected. In Colab, go to Runtime -> Change runtime type -> Select T4 GPU.')

In [ ]:
# Step 2: Mount Drive or Clone Repo
import os
from pathlib import Path

BASE_DIR = Path.cwd()
TRAIN_DIR = BASE_DIR / 'data' / 'raw' / 'fer2013' / 'train'
TEST_DIR = BASE_DIR / 'data' / 'raw' / 'fer2013' / 'test'
MODELS_DIR = BASE_DIR / 'outputs' / 'models'
GRAPHS_DIR = BASE_DIR / 'outputs' / 'graphs'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print('Train Path:', TRAIN_DIR, '| Exists:', TRAIN_DIR.exists())
print('Test Path:', TEST_DIR, '| Exists:', TEST_DIR.exists())

In [ ]:
# Step 3: Compute Balanced Class Weights
counts = {}
class_names = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
for idx, cname in enumerate(class_names):
    counts[idx] = len(list((TRAIN_DIR / cname).glob('*')))

total = sum(counts.values())
num_classes = len(class_names)
class_weights = {idx: total / (num_classes * counts[idx]) for idx in range(num_classes)}

print('Class Distributions & Computed Weights:')
for idx, cname in enumerate(class_names):
    print(f' - {cname:<10s}: {counts[idx]:5d} samples -> Weight: {class_weights[idx]:.3f}')

In [ ]:
# Step 4: Data Augmentation & Datasets
from tensorflow.keras import layers

IMG_SIZE = (48, 48)
BATCH_SIZE = 64

raw_train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=True,
    seed=42
)

raw_test = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=False
)

augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.15)
])

train_ds = raw_train.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_ds = raw_test.prefetch(tf.data.AUTOTUNE)
print('Datasets prepared with prefetching and augmentation.')

In [ ]:
# Step 5: Build Deep Mini-Xception Model
from tensorflow.keras import models, regularizers

def build_model(input_shape=(48, 48, 1), num_classes=7, l2_reg=1e-4):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 255.0)(inputs)

    # Entry block
    x = layers.Conv2D(16, (3, 3), use_bias=False, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(16, (3, 3), use_bias=False, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Residual separable blocks
    for filters in [32, 64, 128, 256]:
        res = layers.Conv2D(filters, (1, 1), strides=2, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(l2_reg))(x)
        res = layers.BatchNormalization()(res)

        x = layers.SeparableConv2D(filters, (3, 3), padding='same', use_bias=False, depthwise_regularizer=regularizers.l2(l2_reg), pointwise_regularizer=regularizers.l2(l2_reg))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)

        x = layers.SeparableConv2D(filters, (3, 3), padding='same', use_bias=False, depthwise_regularizer=regularizers.l2(l2_reg), pointwise_regularizer=regularizers.l2(l2_reg))(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((3, 3), strides=2, padding='same')(x)

        x = layers.add([x, res])
        x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(num_classes, (3, 3), padding='same')(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Activation('softmax')(x)
    return models.Model(inputs, outputs)

model = build_model()
model.summary()

In [ ]:
# Step 6: Compile & Train with Callbacks
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.08),
    metrics=['accuracy']
)

checkpoint_path = str(MODELS_DIR / 'best_emotion_model.keras')

callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=35,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
# Step 7: Plot Accuracy & Loss Curves
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc', lw=2)
plt.plot(history.history['val_accuracy'], label='Val Acc', lw=2)
plt.title('Accuracy vs Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', lw=2)
plt.plot(history.history['val_loss'], label='Val Loss', lw=2)
plt.title('Loss vs Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Step 8: Comprehensive Classification Report & Confusion Matrix
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print('\n' + '='*60)
print('FER2013 Classification Report:')
print('='*60)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Normalized)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()